In [ ]:
!pip install ultralytics wandb roboflow

!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu128

In [11]:
import torch
import wandb
from ultralytics import YOLO
from roboflow import Roboflow

In [ ]:
rf = Roboflow(api_key="MY-API-KEY")

project = rf.workspace("licence-plate-3jqkb").project("license-plate-az47f")
version = project.version(1)
dataset = version.download("yolov8")

print(dataset.location)

In [ ]:
wandb.login()

wandb.init(project="license_plate_detection", job_type="training")

In [ ]:
import yaml
import os
from ultralytics import YOLO

yaml_path = f"{dataset.location}/data.yaml"

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

train_images_path = os.path.join(dataset.location, 'train', 'images')
valid_images_path = os.path.join(dataset.location, 'valid', 'images')

data['train'] = train_images_path
data['val'] = valid_images_path

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project="plates_project",
    name="yolo_train",
    plots=True
)

wandb.finish()